# 1.

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
# Cài đặt thư viện nếu cần
!pip install timm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.4 MB/s eta 0:00:00


In [3]:
import os
import torch
import timm
import numpy as np
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Đường dẫn dữ liệu
data_dir = "/kaggle/input/data-cv-01/data"  # bạn cần unzip dữ liệu vào đây

# Biến đổi ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Tải dữ liệu
dataset = ImageFolder(data_dir, transform=transform)

# Tách train / val
from torch.utils.data import random_split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

Device: cuda


In [4]:
# Dùng ViT từ thư viện timm (pretrained)
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=4)

# Do ảnh là 90x90, cần resize head nếu muốn fine-tune trên input size nhỏ
model.patch_embed.proj = nn.Conv2d(3, model.embed_dim, kernel_size=16, stride=16)

model.to(device)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity(

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct = 0.0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return running_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    running_loss, correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return running_loss / len(loader), correct / len(loader.dataset)

In [6]:
epochs = 20
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")


Epoch 1/20
Train Loss: 0.6784, Accuracy: 0.7244
Val Loss: 0.2980, Accuracy: 0.8940
Epoch 2/20
Train Loss: 0.2616, Accuracy: 0.9088
Val Loss: 0.2195, Accuracy: 0.9204
Epoch 3/20
Train Loss: 0.2054, Accuracy: 0.9272
Val Loss: 0.2024, Accuracy: 0.9308
Epoch 4/20
Train Loss: 0.1682, Accuracy: 0.9390
Val Loss: 0.2558, Accuracy: 0.9122
Epoch 5/20
Train Loss: 0.1443, Accuracy: 0.9508
Val Loss: 0.1615, Accuracy: 0.9433
Epoch 6/20
Train Loss: 0.1352, Accuracy: 0.9526
Val Loss: 0.2123, Accuracy: 0.9260
Epoch 7/20
Train Loss: 0.1129, Accuracy: 0.9602
Val Loss: 0.1577, Accuracy: 0.9455
Epoch 8/20
Train Loss: 0.1064, Accuracy: 0.9640
Val Loss: 0.1448, Accuracy: 0.9472
Epoch 9/20
Train Loss: 0.0940, Accuracy: 0.9654
Val Loss: 0.1324, Accuracy: 0.9572
Epoch 10/20
Train Loss: 0.0949, Accuracy: 0.9645
Val Loss: 0.1626, Accuracy: 0.9472
Epoch 11/20
Train Loss: 0.0924, Accuracy: 0.9662
Val Loss: 0.1223, Accuracy: 0.9619
Epoch 12/20
Train Loss: 0.0657, Accuracy: 0.9771
Val Loss: 0.1511, Accuracy: 0.9498
E

In [7]:
torch.save(model.state_dict(), "/kaggle/working/vit_model_01.pth")